# reduceRegions: a decade of fire detections per district in Riau (Python)

**Fullstack Remote Sensing** · code by **Rifky Nauval Hendrawan, Remote Sensing Specialist** · [Chapter: Server Side Paradigms](https://rifkynhsp-byte.github.io/Fullstack-Remote-Sensing/en/07-server-side.html) · [Bab (Bahasa Indonesia)](https://rifkynhsp-byte.github.io/Fullstack-Remote-Sensing/id/07-server-side.html) · [Open the JavaScript in the Code Editor](https://code.earthengine.google.com/?scriptPath=users/rifkynauvalhsp/Fullstack-Remote-Sensing:ch07_fire_by_regency)

Run the cells from top to bottom. The first cell installs what Colab lacks, fetches the book's data and helpers, and signs you in to Earth Engine with your own Google Cloud project ([how to get one](https://developers.google.com/earth-engine/guides/access)).

*Jalankan sel dari atas ke bawah. Sel pertama memasang paket, mengambil data buku, dan masuk ke Earth Engine dengan proyek Google Cloud Anda sendiri.*

In [ ]:
# @title Setup: packages, book data, Earth Engine sign-in
!pip -q install earthengine-api geemap tabulate
!test -d /content/book || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/rifkynhsp-byte/Fullstack-Remote-Sensing.git /content/book
!git -C /content/book sparse-checkout set tools scripts data   # only what the code needs, not the website
import sys, os; sys.path[:0] = ['/content/book/tools', '/content/book/scripts/py']; os.chdir('/content/book')
__file__ = '/content/book/scripts/py/ch07_fire_by_regency.py'   # the script finds the book's data relative to itself
PROJECT = "your-cloud-project"  # @param {type:"string"}
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT)
_init = ee.Initialize; ee.Initialize = lambda *a, **k: None   # the script must not sign in again

CHAPTER 7 | FIRMS fire detections summed per Riau district with
reduceRegions, and per year for the province.

In [ ]:
# Fullstack Remote Sensing | Code by Rifky Nauval Hendrawan, Remote Sensing Specialist
# MIT licence: free to use and adapt; please keep this credit line.

"""
CHAPTER 7 | FIRMS fire detections summed per Riau district with
reduceRegions, and per year for the province.
"""

import ee
import matplotlib.pyplot as plt

districts = (ee.FeatureCollection("FAO/GAUL/2025/level2")
             .filter(ee.Filter.eq("GAUL1_NAME", "Riau")))
riau = districts.geometry()

firms = (ee.ImageCollection("FIRMS").filterBounds(riau)
         .map(lambda img: img.select("confidence").gte(50).selfMask().rename("fire")
              .copyProperties(img, ["system:time_start"])))

fire_days = (firms.filterDate("2015-01-01", "2025-01-01").count()
             .rename("fire_days").clip(riau))

In [ ]:
def per_district_props(f):
    km2 = f.geometry().area(100).divide(1e6)
    return f.set({"district": f.get("GAUL2_NAME"), "fire_days": f.get("sum"),
                  "per_1000_km2": ee.Number(f.get("sum")).divide(km2).multiply(1000)})


per_district = (fire_days.reduceRegions(collection=districts, reducer=ee.Reducer.sum(),
                                        scale=1000, tileScale=4)
                .map(per_district_props)
                .select(["district", "fire_days", "per_1000_km2"], retainGeometry=False))

In [ ]:
def year_total(y):
    n = (firms.filter(ee.Filter.calendarRange(y, y, "year")).count()
         .reduceRegion(reducer=ee.Reducer.sum(), geometry=riau, scale=1000,
                       maxPixels=1e10, tileScale=4).get("fire"))
    return ee.Feature(None, {"year": y, "fire_days": n})


per_year = ee.FeatureCollection([year_total(y) for y in range(2015, 2025)])

map_image = (fire_days.visualize(min=1, max=30,
                                 palette=["fee391", "fe9929", "cc4c02", "662506"])
             .blend(ee.Image().byte().paint(districts, 1, 1).visualize(palette=["000000"])))

In [ ]:
def plot_districts(df):
    """Two rankings: raw totals and density. They do not agree."""
    df = df.sort_values("per_1000_km2")
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
    axes[0].barh(df["district"], df["fire_days"], color="#cc4c02")
    axes[0].set_xlabel("Fire detection days, 2015 to 2024")
    axes[1].barh(df["district"], df["per_1000_km2"], color="#662506")
    axes[1].set_xlabel("Per 1,000 km²")
    fig.suptitle("Riau districts: total against density", x=0.02, ha="left")
    fig.tight_layout()
    return fig

In [ ]:
def plot_years(df):
    """The province, one bar per year."""
    df = df.sort_values("year")
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(df["year"].astype(int).astype(str), df["fire_days"], color="#cc4c02")
    ax.set_ylabel("Fire detection days")
    ax.set_title("Riau, fire detection days per year", loc="left")
    return fig

In [ ]:
def products():
    return [
        {"kind": "map", "name": "ch07-fire-map", "image": map_image, "region": riau.bounds(),
         "vis": {"bands": ["vis-red", "vis-green", "vis-blue"], "min": 0, "max": 255},
         "classes": [("1 day", "#fee391"), ("10 days", "#fe9929"), ("20 days", "#cc4c02"),
                     ("30+ days", "#662506")],
         "title": "Fire detection days per 1 km pixel, 2015 to 2024",
         "source": "FIRMS (NASA), confidence ≥ 50. Boundaries: FAO GAUL 2025.",
         "caption": "Every 1 km pixel's count of days with a confident fire detection, "
                    "with district boundaries."},
        {"kind": "chart", "name": "ch07-fire-districts", "data": per_district,
         "plot": plot_districts,
         "caption": "reduceRegions returned one row per district in a single call."},
        {"kind": "chart", "name": "ch07-fire-years", "data": per_year, "plot": plot_years,
         "caption": "The province's yearly total."},
        {"kind": "table", "name": "ch07-fire-table", "data": per_district,
         "columns": ["district", "fire_days", "per_1000_km2"], "floatfmt": ("", ".0f", ".1f"),
         "caption": "Per district, 2015 to 2024."},
    ]

### The chapter's maps, charts and tables
The same products the book shows, drawn here. Maps also open as an interactive layer.

In [ ]:
from colab_render import show_all
show_all(products())